# `zemi.toml` capabilities

The module returns a standard `dict`/`list` tree and validates unique names and ZEMI reference existence without resolving their contents.

In [ ]:
# Automatically reload imported modules when their source code changes
%load_ext autoreload
%autoreload 2

# Set the working directory to the ZEMI component root
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

In [ ]:
from zemi import env, toml

CONFIG_PATH = PROJECT_ROOT / 'tests/zemi_toml/test_zemi_toml.toml'
config = toml.load(CONFIG_PATH)

## Plain Python tree

Nested tables remain dictionaries and arrays of tables remain lists. The consuming module, such as `zemi.arsenal`, creates the named domain tree.

In [ ]:
assert type(config) is dict
assert type(config['arsenal']) is dict
assert type(config['arsenal']['llamas']) is list
primary = config['arsenal']['llamas'][0]
qwen = primary['models'][0]
assistant = qwen['assistants'][0]
primary['name'], qwen['name'], assistant['name']

## ZEMI references are preserved

`@comp/...` and `@inst/...` are checked for existence but remain unchanged strings. Only the direct consumer reads file contents.

In [ ]:
prefix = assistant['prefix']
assert prefix == '@comp/tests/zemi_toml/prefixes/qwen-system.md'
assert config['non_text_reference'] == "@comp/.zemicomp"
prefix

## Validation

Duplicate non-empty `name` values in one array and references to missing paths are rejected. Temporary files are created only in `@inst/_tmp`.

In [ ]:
from tempfile import TemporaryDirectory

env.path.tmp.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=env.path.tmp) as directory:
    duplicate = Path(directory) / 'duplicate.toml'
    duplicate.write_text(
        "[[items]]\nname = 'same'\n[[items]]\nname = 'same'\n",
        encoding='utf-8',
    )
    try:
        toml.load(duplicate)
    except ValueError as error:
        assert "duplicate name 'same'" in str(error)
    else:
        raise AssertionError('Expected ValueError')

with TemporaryDirectory(dir=env.path.tmp) as directory:
    missing = Path(directory) / 'missing.toml'
    missing.write_text(
        "reference = '@comp/does-not-exist-for-toml-test.txt'\n",
        encoding='utf-8',
    )
    try:
        toml.load(missing)
    except FileNotFoundError:
        pass
    else:
        raise AssertionError('Expected FileNotFoundError')